# Libraries

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

import glob
import hashlib  # noqa: F401
import re
import numpy as np
import pandas as pd
import janitor  # noqa: F401
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(_root / "src"))
from tfm.utils import ROOT_PATH, DATA_PATH, RAW_PATH, PROCESSED_PATH
from tfm.preprocessing import clean_subject_text, hash_id, normalize_product, CATEGORY_RULES

np.random.seed(42)

# Load Data

In [ ]:
low_files = glob.glob(str(RAW_PATH / "*LOW*.csv"))
cpc_files = glob.glob(str(RAW_PATH / "*CPC*.csv"))
# Read and concatenate all CSVs
low_data = pd.concat(
    [pd.read_csv(archivo, sep=";", low_memory=False) for archivo in low_files],
    ignore_index=True,
)
cpc_data = pd.concat(
    [pd.read_csv(archivo, sep=";", low_memory=False) for archivo in cpc_files],
    ignore_index=True,
)

# Normalize column names and filter CPC data
cpc_data = cpc_data.clean_names()
low_data = low_data.clean_names()
cpc_data = cpc_data[cpc_data["unnamed_22"] == "CPL"]
cpc_data = cpc_data.rename(columns={"precio": "cpl"})

# Select only the relevant features
FEATURES = [
    "email",
    "fecha_de_nacimiento",
    "sexo",
    "codigo_postal",
    "event_timestamp",
    "sector",
    "producto",
    "marca_anunciante",
    "empresa_prov",
    "subject",
    "event_type",
    "cpl",
]

low_data = low_data[FEATURES]
cpc_data = cpc_data[FEATURES]
# Merge datasets
df_raw = pd.concat([low_data, cpc_data], ignore_index=False)
# Balance 1:1 using ALL clicks available in the raw data + an equal number of non-clicks.
# Note: the model's EFFECTIVE training prior is rho_train~0.238 (not 0.5), because the
# target is defined as click vs open (the 'ignored' events are discarded) and after the
# deduplication/filters. It is measured empirically in notebook 07 and corrected back to
# the real prior rho_real~0.02 with src/tfm/calibration.py (correct_prior takes rho_train explicitly).
_clicks     = df_raw[df_raw["event_type"] == "click"]
_non_clicks = df_raw[df_raw["event_type"] != "click"].sample(n=len(_clicks), random_state=42)
df_raw = pd.concat([_clicks, _non_clicks]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Balance raw  →  clicks: {len(_clicks):,}  |  no-clicks: {len(_non_clicks):,}  |  total: {len(df_raw):,}")

# Clean data

In [ ]:
# Copy dataframe to avoid modifying the original
df_clean = df_raw.copy()
# Rename columns for consistency
df_clean = df_clean.rename(
    columns={
        "fecha_de_nacimiento": "birthday",
        "codigo_postal": "cp_num",
        "event_timestamp": "timestamp",
        "marca_anunciante": "brand",
        "empresa_prov": "company",
        "sexo": "gender",
        "producto": "product",
    },
    inplace=False,
)


df_clean = df_clean.assign(
    birthday=pd.to_datetime(df_clean["birthday"], errors="coerce"),
    timestamp=pd.to_datetime(df_clean["timestamp"], errors="coerce"),
)

cat_cols = ["gender", "sector", "product", "brand", "company", "event_type"]
df_clean[cat_cols] = df_clean[cat_cols].astype(
    "string"
)  # Later we change the type of variables to category, but we need to clean them first

str_cols = ["email", "subject", "cp_num"]
df_clean[str_cols] = df_clean[str_cols].astype("string")

# cpl as numeric; cp_num is kept as a string zero-padded to 5 digits
df_clean["cpl"] = pd.to_numeric(df_clean["cpl"], errors="coerce").round(0).astype("Int64")
df_clean["cp_num"] = (
    pd.to_numeric(df_clean["cp_num"], errors="coerce")
    .apply(lambda x: str(int(x)).zfill(5) if pd.notna(x) else pd.NA)
    .astype("string")
)
# Drop rows with missing values in critical columns and remove duplicates
df_clean = df_clean.dropna(subset=["email", "cp_num"], inplace=False)
df_clean = df_clean.drop_duplicates()


def clean_text(s):
    if pd.isna(s):
        return s
    return str(s).strip().lower()


df_clean["email_clean"] = df_clean["email"].apply(clean_text)


df_clean["id_user"] = df_clean["email_clean"].apply(hash_id)


# We compute the age at a FIXED reference DATE (the data extraction date),
# so that the result is REPRODUCIBLE and does not depend on the day the notebook is run.
FECHA_REF = pd.Timestamp("2026-01-05")
df_clean["age"] = (FECHA_REF - df_clean["birthday"]).dt.days // 365.25
df_clean["age"] = pd.to_numeric(df_clean["age"], errors="coerce")
df_clean = df_clean[(df_clean["age"] >= 18) & (df_clean["age"] <= 120)]
df_clean["age"] = df_clean["age"].astype("Int64")
# We deduplicate by (email, subject) keeping the highest-priority event.
# Only the click is prioritised (1); open and ignored are treated equally (0).
priority_map = {"ignored": 0, "open": 0, "click": 1}
df_clean["priority"] = df_clean["event_type"].map(priority_map).fillna(-1)
df_clean = df_clean.sort_values("priority", ascending=False)
df_clean = df_clean.drop_duplicates(subset=["email", "subject"], keep="first")
df_clean = df_clean.drop(columns="priority")


# Separate concepts 

In [ ]:
users = df_clean[
    ["id_user", "email_clean", "birthday", "age", "gender", "cp_num"]
].drop_duplicates(subset=["id_user"])

df_clean["subject_clean"] = df_clean["subject"].apply(clean_subject_text)


products = df_clean[
    ["subject_clean", "sector", "product", "brand", "company", "cpl"]
].drop_duplicates(subset=["subject_clean"])
products["id_product"] = products["subject_clean"].apply(
    lambda x: hashlib.sha256(x.encode()).hexdigest()
)
products["brand"] = (
    products["brand"]
    .str.lower()
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
    .str.strip()
)

replace_dict = {
    "movistarplus": "movistar plus",
    "marcablanca": "marca blanca",
    "costacrucero": "costa crucero",
    "lineadirecta": "linea directa",
    "masmovil": "mas movil",
    "helvetria": "helvetia",
    "aldeassuicidio": "aldeas infantiles",
}

products["brand"] = products["brand"].apply(clean_text)
products["brand"] = products["brand"].replace(replace_dict)


In [ ]:
replace_dict = {
    "digitalmmedia": "digital media",
    "digitalmedia": "digital media",
    "ad pepper": "ad pepper",
    "adpepper": "ad pepper",
    "startend/feebbo": "startend",
    "adviceme/digitalmedia": "adviceme",
}

products["company"] = products["company"].apply(clean_text)

products["company"] = products["company"].replace(replace_dict)

In [ ]:
replace_dict = {
    "seguros hogar": "seguros",
}

products["sector"] = products["sector"].apply(clean_text)

products["sector"] = products["sector"].replace(replace_dict)

In [ ]:
products["product_new"] = products.apply(normalize_product, axis=1)
products[["subject_clean", "product", "product_new"]]


In [ ]:
events = df_clean[["email_clean", "subject_clean", "event_type", "timestamp"]]
events = pd.merge(
    events, products[["subject_clean", "id_product"]], on="subject_clean", how="left"
)
events = pd.merge(
    events, users[["email_clean", "id_user"]], on="email_clean", how="left"
)
# 1. We concatenate the columns as text (handling nulls just in case)
texto_combinado = events["subject_clean"].fillna("").astype(str) + events[
    "email_clean"
].fillna("").astype(str)

# 2. We generate the hash natively and directly
# We convert the resulting number to its hexadecimal representation so it has an ID-like format
events["id_event"] = pd.util.hash_pandas_object(texto_combinado, index=False).apply(hex)

In [ ]:
events.value_counts("event_type")
events = events[["id_event", "timestamp", "event_type", "id_product", "id_user"]]

In [ ]:
users.to_csv(DATA_PATH / "processed" / "users_base.csv", index=False)
products.to_csv(DATA_PATH / "processed" / "products.csv", index=False)
events.to_csv(DATA_PATH / "processed" / "events.csv", index=False)